# Fetch GPS Track History

**Purpose:** pull raw GPS pings for every tracker over a given date range and save to `data/gps_data_<label>.csv`. This is the heavy, long-running pull (one API call per tracker, with a small delay between calls) — run it per month/period rather than re-pulling the full history each time.

Requires `data/tracker_list.csv` to already exist (run `fetch_tracker_list.ipynb` first).

In [ ]:
import time
import sys
sys.path.append("../..")

import pandas as pd

from gps_lib import navixy_api, io_utils

In [ ]:
# --- Parameters: edit these for the period you want to pull ---
FROM_TIME = "2025-07-01 00:00:00"
TO_TIME = "2025-07-31 23:59:59"
OUTPUT_FILENAME = "gps_data_2025-7.csv"
REQUEST_DELAY_SEC = 0.5

In [ ]:
hash_token = navixy_api.authenticate()
tracker_list_df = io_utils.load_tracker_list()
len(tracker_list_df)

In [ ]:
all_tracks, failed_trackers = [], []

for i, tracker_id in enumerate(tracker_list_df["id"], 1):
    print(f"[{i}/{len(tracker_list_df)}] tracker {tracker_id}...")
    track_df = navixy_api.get_track_read(hash_token, tracker_id, FROM_TIME, TO_TIME)
    if track_df is not None and not track_df.empty:
        track_df["tracker_id"] = tracker_id
        all_tracks.append(track_df)
        print(f"  collected {len(track_df)} points")
    else:
        print("  no data")
        failed_trackers.append(tracker_id)
    time.sleep(REQUEST_DELAY_SEC)

final_df = pd.concat(all_tracks, ignore_index=True) if all_tracks else pd.DataFrame()
print(f"\nCollected {len(final_df)} points from {len(all_tracks)} trackers; "
      f"{len(failed_trackers)} failed: {failed_trackers}")

In [ ]:
out_path = io_utils.save_csv(final_df, OUTPUT_FILENAME)
print("Saved:", out_path)